# Bike Sharing Demand Prediction

## Data Science Mini Project

**Objective:** Predict the total number of bikes rented using weather, seasonal, date and time-related features.

**Dataset:** UCI Bike Sharing Dataset (hourly data)

This notebook performs data loading, cleaning, exploratory data analysis (EDA), feature preparation, model building, evaluation, model comparison, and conclusion.

## 1. Import Libraries

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

## 2. Download and Load the Dataset

The official UCI repository provides the Bike Sharing Dataset. The code below downloads the dataset directly in Google Colab and extracts `hour.csv`.

In [ ]:
# Install the UCI dataset helper
!pip -q install ucimlrepo

from ucimlrepo import fetch_ucirepo

# Fetch UCI Bike Sharing dataset
bike_sharing = fetch_ucirepo(id=275)

# Combine features and target so the required CSV can be saved locally
df = bike_sharing.data.features.copy()
df["cnt"] = bike_sharing.data.targets["cnt"].values

# Save the dataset for GitHub submission
df.to_csv("bike_sharing_hour.csv", index=False)

print("Dataset saved as bike_sharing_hour.csv")
print("Shape:", df.shape)
df.head()

## 3. Understand the Dataset

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
df.describe().T

## 4. Data Cleaning and Feature Preparation

`casual` and `registered` are not used as predictors because together they form the total rental count `cnt`. Keeping them would cause target leakage. The record identifier is also removed.

In [ ]:
data = df.copy()

# Convert date column
data["dteday"] = pd.to_datetime(data["dteday"])

# Extract useful date features
data["day"] = data["dteday"].dt.day
data["date_month"] = data["dteday"].dt.month
data["date_year"] = data["dteday"].dt.year

# Remove identifier, raw date, and leakage columns
data = data.drop(columns=["instant", "dteday", "casual", "registered"])

print("Cleaned shape:", data.shape)
data.head()

## 5. Exploratory Data Analysis

In [ ]:
# Target distribution
plt.figure(figsize=(8, 5))
plt.hist(data["cnt"], bins=40)
plt.title("Distribution of Total Bike Rentals")
plt.xlabel("Total Rentals (cnt)")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Rentals by hour
hour_avg = data.groupby("hr")["cnt"].mean()
plt.figure(figsize=(9, 5))
hour_avg.plot(kind="line", marker="o")
plt.title("Average Bike Rentals by Hour")
plt.xlabel("Hour")
plt.ylabel("Average Rentals")
plt.grid(True)
plt.show()

In [ ]:
# Rentals by weather situation
weather_avg = data.groupby("weathersit")["cnt"].mean()
plt.figure(figsize=(8, 5))
weather_avg.plot(kind="bar")
plt.title("Average Bike Rentals by Weather Situation")
plt.xlabel("Weather Situation")
plt.ylabel("Average Rentals")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Correlation with target for numeric columns
correlations = data.select_dtypes(include=np.number).corr()["cnt"].sort_values(ascending=False)
print(correlations)

## 6. Prepare Data for Machine Learning

In [ ]:
X = data.drop(columns=["cnt"])
y = data["cnt"]

# Treat coded categorical variables as categorical
categorical_features = ["season", "yr", "mnth", "hr", "holiday", "weekday", "workingday", "weathersit"]
numerical_features = [c for c in X.columns if c not in categorical_features]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numerical_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features)
    ]
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 7. Train Multiple Regression Models

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=150, random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=150, random_state=42
    )
}

results = []
trained_models = {}

for name, model in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    pipe.fit(X_train, y_train)
    predictions = pipe.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2 Score": r2
    })
    trained_models[name] = pipe

results_df = pd.DataFrame(results).sort_values("R2 Score", ascending=False)
results_df

## 8. Compare Model Performance

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(results_df["Model"], results_df["R2 Score"])
plt.title("Model Comparison - R2 Score")
plt.xlabel("Model")
plt.ylabel("R2 Score")
plt.xticks(rotation=15)
plt.show()

best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]
print("Best Model:", best_model_name)
print(results_df.iloc[0])

## 9. Save the Best Model

In [ ]:
joblib.dump(best_model, "best_bike_sharing_model.pkl")
print("Saved: best_bike_sharing_model.pkl")

## 10. Example Prediction

In [ ]:
sample = X_test.iloc[[0]]
predicted_value = best_model.predict(sample)[0]
actual_value = y_test.iloc[0]

print(f"Actual rental count: {actual_value:.0f}")
print(f"Predicted rental count: {predicted_value:.0f}")

## 11. Final Conclusion

The project compares Linear Regression, Random Forest, and Gradient Boosting for predicting bike rental demand. The best model is selected using the highest R2 score on the test set. MAE and RMSE are also reported to measure prediction error.

The analysis shows that bike rental demand is influenced by time, season, weather and environmental conditions. The final trained model is saved as `best_bike_sharing_model.pkl` for submission.

## 12. Files Created for GitHub

After running all cells, the notebook creates:

- `bike_sharing_hour.csv` — dataset used by the project
- `best_bike_sharing_model.pkl` — saved best-performing model
- This notebook (`.ipynb`) — complete implementation
- `README.md` — project documentation
